# Choice of Attention Layers

Thesis: *Experimental Setup*, section *Measuring Attention Dominance*, subsections *Choice of Attention Layers* and *The Dominance Metric*.

The control images were generated with `src/generation/run_attention_controls.py`.

**Requirements**
- Environment: `easy-language` (`environment-easy-language.yml`)
- Packages: pandas, transformers, tokenizers, huggingface_hub
- Data: `data/attention_controls/size/size_control_results.csv`, `data/attention_controls/length/length_control_results.csv` (in this repository)
- Network: the tokenizer of `black-forest-labs/FLUX.2-klein-4B` at a pinned revision, a few MB, public
- GPU: not needed

**Writes:** `exports/control_tokenization.csv`, `exports/size_control_*.csv`, `exports/length_control_*.csv`

In [1]:
import os
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "corpus").is_dir())
os.chdir(ROOT)
for folder in ("generation",):
    sys.path.insert(0, str(ROOT / "src" / folder))


import pandas as pd
from transformers import Qwen2TokenizerFast

from control_stimuli import (
    DOMINANCE_CONTROL_PROMPTS,
    LENGTH_CONTROL_ARTICLES,
    LENGTH_CONTROL_FRAME,
    LENGTH_CONTROL_ORDERS,
    LENGTH_CONTROL_PAIRS,
)
from token_alignment import FLUX_REVISION, MODEL_ID, get_word_groups
from IPython.display import display

## Token counts of the control names

Token counts of the control names. They define the two pair groups of the size control (same token count, or the larger object with the shorter name) and the length difference of the length control.

In [2]:
OUT_PATH = ROOT / "exports" / "control_tokenization.csv"
OUT_FIELDS = [
    "control", "pair", "role", "order", "object_name", "n_tokens", "token_pieces", "prompt",
]

UMLAUT_VARIANTS = [("Feuerlöscher", "Feuerloescher")]


def word_pieces(tokenizer, prompt: str, target: str) -> tuple[int, list[str]] | None:
    """Token count and surface pieces of `target` inside `prompt`, in the encoder's coordinates."""
    words = get_word_groups(tokenizer, prompt)
    bare_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
    from token_alignment import templated_prompt_alignment

    _, template_offset, _ = templated_prompt_alignment(tokenizer, prompt)
    for surface, token_indices in words:
        if surface.strip().lower() == target.strip().lower():
            pieces = tokenizer.convert_ids_to_tokens(
                [bare_ids[i - template_offset] for i in token_indices]
            )
            return len(token_indices), pieces
    return None


def length_control_prompts() -> list[tuple[dict, str, str, str]]:
    """(pair, order, prompt, first_name) for every realised length-control prompt."""
    out = []
    for pair in LENGTH_CONTROL_PAIRS:
        article = LENGTH_CONTROL_ARTICLES[pair["short"]]
        for order in LENGTH_CONTROL_ORDERS:
            first, second = (
                (pair["short"], pair["long"]) if order == "short_first" else (pair["long"], pair["short"])
            )
            prompt = LENGTH_CONTROL_FRAME.format(
                art_a=article, a=first, art_b=article.lower(), b=second, ground=pair["ground"]
            )
            out.append((pair, order, prompt, first))
    return out

In [3]:
tokenizer = Qwen2TokenizerFast.from_pretrained(MODEL_ID, subfolder="tokenizer", revision=FLUX_REVISION)

In [4]:
rows = []
size_pairs = []

# --- size-dominance control ---
print("=== Size-dominance control: are the pairs length-comparable? ===")
for entry in DOMINANCE_CONTROL_PROMPTS:
    prompt = entry["prompt"]
    got = {}
    for role in ("big", "small"):
        name = entry[role]
        res = word_pieces(tokenizer, prompt, name)
        if res is None:
            print(f"  WARNING: {name!r} not found as a word in {prompt!r}")
            continue
        n, pieces = res
        got[role] = (name, n, pieces)
        rows.append({
            "control": "size_dominance", "pair": f"{entry['big']}/{entry['small']}",
            "role": role, "order": "", "object_name": name,
            "n_tokens": n, "token_pieces": "|".join(pieces), "prompt": prompt,
        })
    if len(got) == 2:
        (bn, bt, bp), (sn, st, sp) = got["big"], got["small"]
        size_pairs.append({"pair": f"{bn}/{sn}", "big tokens": bt, "small tokens": st,
                           "delta": bt - st, "big pieces": " | ".join(bp),
                           "small pieces": " | ".join(sp)})
display(pd.DataFrame(size_pairs).set_index("pair"))

=== Size-dominance control: are the pairs length-comparable? ===


,big tokens,small tokens,delta,big pieces,small pieces
pair,,,,,
Elefant/Vogel,2,2,0,ĠEle | fant,ĠVog | el
Lastwagen/Fahrrad,3,3,0,ĠLast | w | agen,ĠFah | rr | ad
Haus/Blume,1,2,-1,ĠHaus,ĠBl | ume
Baum/Pilz,1,2,-1,ĠBaum,ĠPil | z
Bus/Roller,1,1,0,ĠBus,ĠRoller
Schiff/Ente,1,2,-1,ĠSchiff,ĠEnt | e


In [5]:
# --- length control ---
print("\n=== Length control: token counts of the two names, per realised prompt ===")
len_rows = []
for pair, order, prompt, _first in length_control_prompts():
    line = {"pair": f"{pair['short']}/{pair['long']}", "order": order, "prompt": prompt}
    for role in ("short", "long"):
        name = pair[role]
        res = word_pieces(tokenizer, prompt, name)
        if res is None:
            print(f"  WARNING: {name!r} not found as a word in {prompt!r}")
            continue
        n, pieces = res
        line[role] = (name, n, pieces)
        rows.append({
            "control": "length", "pair": f"{pair['short']}/{pair['long']}",
            "role": role, "order": order, "object_name": name,
            "n_tokens": n, "token_pieces": "|".join(pieces), "prompt": prompt,
        })
    len_rows.append(line)

display(pd.DataFrame([
    {"pair": line["pair"], "order": line["order"], "prompt": line["prompt"],
     "short tokens": line["short"][1], "long tokens": line["long"][1],
     "ratio long/short": round(line["long"][1] / line["short"][1], 2),
     "short pieces": " | ".join(line["short"][2]), "long pieces": " | ".join(line["long"][2])}
    for line in len_rows if "short" in line and "long" in line]).set_index(["pair", "order"]))

print("\n=== Order invariance of the token counts ===")
by_name: dict[str, set[int]] = {}
for r in rows:
    if r["control"] == "length":
        by_name.setdefault(r["object_name"], set()).add(r["n_tokens"])
display(pd.DataFrame([{"name": name, "token count": "stable" if len(counts) == 1 else f"VARIES {sorted(counts)}"}
                      for name, counts in sorted(by_name.items())]).set_index("name"))


=== Length control: token counts of the two names, per realised prompt ===


prompt  \
pair                  order                                                            
Eimer/Feuerlöscher    short_first  Ein Eimer und ein Feuerlöscher stehen nebenein...   
                      long_first   Ein Feuerlöscher und ein Eimer stehen nebenein...   
Vase/Kaffeemaschine   short_first  Eine Vase und eine Kaffeemaschine stehen neben...   
                      long_first   Eine Kaffeemaschine und eine Vase stehen neben...   
Koffer/Werkzeugkasten short_first  Ein Koffer und ein Werkzeugkasten stehen neben...   
                      long_first   Ein Werkzeugkasten und ein Koffer stehen neben...   

                                   short tokens  long tokens  \
pair                  order                                    
Eimer/Feuerlöscher    short_first             2            4   
                      long_first              2            4   
Vase/Kaffeemaschine   short_first             2            4   
                      long_first              2            4   
Koffer/Werkzeugkasten short_first             2            5   
                      long_first              2            5   

                                   ratio long/short short pieces  \
pair                  order                                        
Eimer/Feuerlöscher    short_first               2.0    ĠE | imer   
                      long_first                2.0    ĠE | imer   
Vase/Kaffeemaschine   short_first               2.0     ĠV | ase   
                      long_first                2.0     ĠV | ase   
Koffer/Werkzeugkasten short_first               2.5   ĠK | offer   
                      long_first                2.5   ĠK | offer   

                                                   long pieces  
pair                  order                                     
Eimer/Feuerlöscher    short_first      ĠFe | uer | lÃ¶s | cher  
                      long_first       ĠFe | uer | lÃ¶s | cher  
Vase/Kaffeemaschine   short_first     ĠK | affe | emas | chine  
                      long_first      ĠK | affe | emas | chine  
Koffer/Werkzeugkasten short_first  ĠWerk | zeug | k | ast | en  
                      long_first   ĠWerk | zeug | k | ast | en


=== Order invariance of the token counts ===


,token count
name,
Eimer,stable
Feuerlöscher,stable
Kaffeemaschine,stable
Koffer,stable
Vase,stable
Werkzeugkasten,stable


In [6]:
# --- umlaut cost ---
print("\n=== Spelling variants (the alternative is measured only, no run uses it) ===")
variants = []
for in_use, alternative in UMLAUT_VARIANTS:
    pair = next((p for p in LENGTH_CONTROL_PAIRS if p["long"] == in_use), None)
    if pair is None:
        continue
    article = LENGTH_CONTROL_ARTICLES[pair["short"]]
    for variant in (in_use, alternative):
        prompt = LENGTH_CONTROL_FRAME.format(
            art_a=article, a=pair["short"], art_b=article.lower(), b=variant,
            ground=pair["ground"],
        )
        res = word_pieces(tokenizer, prompt, variant)
        if res is None:
            print(f"  WARNING: {variant!r} not found as a word in {prompt!r}")
            continue
        n, pieces = res
        variants.append({"spelling": variant, "tokens": n, "pieces": " | ".join(pieces)})
        rows.append({
            "control": "spelling_variant", "pair": f"{pair['short']}/{in_use}",
            "role": "long", "order": "", "object_name": variant,
            "n_tokens": n, "token_pieces": "|".join(pieces), "prompt": prompt,
        })

display(pd.DataFrame(variants).set_index("spelling"))

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
pd.DataFrame(rows, columns=OUT_FIELDS).to_csv(OUT_PATH, index=False, lineterminator="\r\n")
print(f"\nwrote {len(rows)} rows -> {OUT_PATH.relative_to(ROOT)}")


=== Spelling variants (the alternative is measured only, no run uses it) ===


,tokens,pieces
spelling,,
Feuerlöscher,4,ĠFe | uer | lÃ¶s | cher
Feuerloescher,5,ĠFe | uer | lo | es | cher



wrote 26 rows -> exports/control_tokenization.csv


## The size and the length control

The size control selects the dual-stream blocks: under the per-token estimator, blocks 2 and 4 order every length-matched stimulus by depicted size, the other blocks ten of fifteen. The length control shows that dividing by the token count overshoots: at block 4 the longer name receives about 0.6 times the per-token share of the shorter one.

In [7]:
SIZE_CSV = ROOT / "data" / "attention_controls" / "size" / "size_control_results.csv"
LENGTH_CSV = ROOT / "data" / "attention_controls" / "length" / "length_control_results.csv"
OUT = ROOT / "exports"

LENGTH_MATCHED = ["Elefant/Vogel", "Lastwagen/Fahrrad", "Bus/Roller"]
COUNTER_ALIGNED = ["Haus/Blume", "Baum/Pilz", "Schiff/Ente"]


def size_group(pair: str) -> str:
    if pair in LENGTH_MATCHED:
        return "length_matched"
    if pair in COUNTER_ALIGNED:
        return "counter_aligned"
    return "ungrouped"


def analyse_size() -> tuple[pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(SIZE_CSV)
    df["group"] = df["pair"].map(size_group)
    df["win_sum"] = df["big_sum"] > df["small_sum"]
    df["win_per_token"] = df["big_per_token"] > df["small_per_token"]

    by_group = (
        df.groupby(["group", "layer_idx"])
        .agg(
            n=("win_sum", "size"),
            wins_sum=("win_sum", "sum"),
            wins_per_token=("win_per_token", "sum"),
            mean_ratio_sum=("ratio_sum", "mean"),
            mean_ratio_per_token=("ratio_per_token", "mean"),
        )
        .reset_index()
    )
    by_pair = (
        df.groupby(["group", "pair", "layer_idx"])
        .agg(
            n=("win_sum", "size"),
            n_tokens_big=("n_tokens_big", "first"),
            n_tokens_small=("n_tokens_small", "first"),
            wins_sum=("win_sum", "sum"),
            wins_per_token=("win_per_token", "sum"),
            mean_ratio_sum=("ratio_sum", "mean"),
            mean_ratio_per_token=("ratio_per_token", "mean"),
        )
        .reset_index()
    )
    return by_group, by_pair


def classify(d_sum: float, d_pt: float) -> str:
    """How the per-token estimator changes the deviation from the null (ratio 1)."""
    if (d_sum > 0) != (d_pt > 0):
        return "umgekehrt"
    return "verkleinert" if abs(d_pt) < abs(d_sum) else "nicht verkleinert"


def analyse_length() -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    df = pd.read_csv(LENGTH_CSV)
    df["win_sum"] = df["long_sum"] > df["short_sum"]
    df["win_per_token"] = df["long_per_token"] > df["short_per_token"]

    agg = dict(
        n=("win_sum", "size"),
        wins_sum=("win_sum", "sum"),
        wins_per_token=("win_per_token", "sum"),
        mean_diff_sum=("diff_sum", "mean"),
        mean_diff_per_token=("diff_per_token", "mean"),
        mean_ratio_sum=("ratio_sum", "mean"),
        mean_ratio_per_token=("ratio_per_token", "mean"),
    )
    pooled = df.groupby("layer_idx").agg(**agg).reset_index()
    by_order = df.groupby(["order", "layer_idx"]).agg(**agg).reset_index()
    by_pair = df.groupby(["pair", "layer_idx"]).agg(**agg).reset_index()

    for frame in (pooled, by_order, by_pair):
        frame["dev_sum"] = frame["mean_ratio_sum"] - 1.0
        frame["dev_per_token"] = frame["mean_ratio_per_token"] - 1.0
        frame["effect_of_per_token"] = [
            classify(a, b) for a, b in zip(frame["dev_sum"], frame["dev_per_token"])
        ]
        frame["residual_share_of_dev"] = [
            (abs(b) / abs(a)) if a else None for a, b in zip(frame["dev_sum"], frame["dev_per_token"])
        ]
    return pooled, by_order, by_pair

In [8]:
OUT.mkdir(exist_ok=True)

size_group_tab, size_pair_tab = analyse_size()
len_pooled, len_by_order, len_by_pair = analyse_length()

size_group_tab.to_csv(OUT / "size_control_by_group.csv", index=False)
size_pair_tab.to_csv(OUT / "size_control_by_pair.csv", index=False)
len_pooled.to_csv(OUT / "length_control_pooled.csv", index=False)
len_by_order.to_csv(OUT / "length_control_by_order.csv", index=False)
len_by_pair.to_csv(OUT / "length_control_by_pair.csv", index=False)

print("=== SIZE CONTROL: big > small, per layer and estimator, by pair group ===")
for group in ("length_matched", "counter_aligned", "ungrouped"):
    sub = size_group_tab[size_group_tab["group"] == group]
    if sub.empty:
        continue
    pairs = LENGTH_MATCHED if group == "length_matched" else COUNTER_ALIGNED
    print(f"\n-- {group} ({', '.join(pairs)}) --")
    display(sub.drop(columns=["group"]).round(4))

print("\n=== SIZE CONTROL: per pair ===")
display(size_pair_tab.round(4))

print("\n=== LENGTH CONTROL: long > short, pooled over both orders ===")
display(len_pooled.round(6))

print("\n=== LENGTH CONTROL: split by prompt order ===")
display(len_by_order.round(6))

print("\n=== LENGTH CONTROL: per pair ===")
display(len_by_pair.round(6))

print(f"\nwrote 5 tables -> {OUT.relative_to(ROOT)}/size_control_*.csv, length_control_*.csv")

=== SIZE CONTROL: big > small, per layer and estimator, by pair group ===

-- length_matched (Elefant/Vogel, Lastwagen/Fahrrad, Bus/Roller) --


,layer_idx,n,wins_sum,wins_per_token,mean_ratio_sum,mean_ratio_per_token
5,0,15,10,10,1.1088,1.1088
6,1,15,10,10,1.7377,1.7377
7,2,15,15,15,1.9524,1.9524
8,3,15,10,10,1.3101,1.3101
9,4,15,15,15,2.3291,2.3291



-- counter_aligned (Haus/Blume, Baum/Pilz, Schiff/Ente) --


,layer_idx,n,wins_sum,wins_per_token,mean_ratio_sum,mean_ratio_per_token
0,0,15,5,15,0.8488,1.6977
1,1,15,15,15,2.5880,5.1761
2,2,15,13,15,1.5037,3.0074
3,3,15,5,13,0.7479,1.4958
4,4,15,15,15,2.8241,5.6482



=== SIZE CONTROL: per pair ===


,group,pair,layer_idx,n,n_tokens_big,n_tokens_small,wins_sum,wins_per_token,mean_ratio_sum,mean_ratio_per_token
0,counter_aligned,Baum/Pilz,0,5,1,2,5,5,1.0496,2.0992
1,counter_aligned,Baum/Pilz,1,5,1,2,5,5,2.1889,4.3778
2,counter_aligned,Baum/Pilz,2,5,1,2,5,5,2.1158,4.2315
3,counter_aligned,Baum/Pilz,3,5,1,2,5,5,1.1323,2.2646
4,counter_aligned,Baum/Pilz,4,5,1,2,5,5,4.0449,8.0897
5,counter_aligned,Haus/Blume,0,5,1,2,0,5,0.5989,1.1978
6,counter_aligned,Haus/Blume,1,5,1,2,5,5,2.6562,5.3125
7,counter_aligned,Haus/Blume,2,5,1,2,3,5,1.0298,2.0597
8,counter_aligned,Haus/Blume,3,5,1,2,0,4,0.5598,1.1197
9,counter_aligned,Haus/Blume,4,5,1,2,5,5,1.6771,3.3542


... (3 outputs left out)

,order,layer_idx,n,wins_sum,wins_per_token,mean_diff_sum,mean_diff_per_token,mean_ratio_sum,mean_ratio_per_token,dev_sum,dev_per_token,effect_of_per_token,residual_share_of_dev
0,long_first,0,15,15,5,0.019354,-0.000396,2.272566,1.016233,1.272566,0.016233,verkleinert,0.012757
1,long_first,1,15,15,10,0.018945,0.001180,2.654378,1.254907,1.654378,0.254907,verkleinert,0.154080
2,long_first,2,15,15,10,0.007950,0.000058,2.190763,1.000898,1.190763,0.000898,verkleinert,0.000754
3,long_first,3,15,15,7,0.003041,0.000102,2.333352,1.084247,1.333352,0.084247,verkleinert,0.063184
4,long_first,4,15,13,0,0.001953,-0.001503,1.272939,0.595016,0.272939,-0.404984,umgekehrt,1.483788
5,short_first,0,15,15,0,0.012159,-0.002295,1.670136,0.772286,0.670136,-0.227714,umgekehrt,0.339802
6,short_first,1,15,0,0,-0.002365,-0.006267,0.900472,0.424153,-0.099528,-0.575847,nicht verkleinert,5.785753
7,short_first,2,15,11,0,0.003217,-0.001863,1.331546,0.608774,0.331546,-0.391226,umgekehrt,1.180007
8,short_first,3,15,9,0,0.000693,-0.000825,1.332738,0.596362,0.332738,-0.403638,umgekehrt,1.213080
9,short_first,4,15,10,0,0.001683,-0.001579,1.279434,0.585921,0.279434,-0.414079,umgekehrt,1.481848



=== LENGTH CONTROL: per pair ===


,pair,layer_idx,n,wins_sum,wins_per_token,mean_diff_sum,mean_diff_per_token,mean_ratio_sum,mean_ratio_per_token,dev_sum,dev_per_token,effect_of_per_token,residual_share_of_dev
0,Eimer/Feuerlöscher,0,10,10,0,0.011179,-0.001898,1.621663,0.810832,0.621663,-0.189168,umgekehrt,0.304294
1,Eimer/Feuerlöscher,1,10,5,5,0.004927,-0.001670,1.700190,0.850095,0.700190,-0.149905,umgekehrt,0.214092
2,Eimer/Feuerlöscher,2,10,6,0,0.000903,-0.001610,1.167770,0.583885,0.167770,-0.416115,umgekehrt,2.480269
3,Eimer/Feuerlöscher,3,10,9,1,0.001257,-0.000453,1.520955,0.760478,0.520955,-0.239522,umgekehrt,0.459775
4,Eimer/Feuerlöscher,4,10,3,0,-0.001137,-0.002154,0.876906,0.438453,-0.123094,-0.561547,nicht verkleinert,4.561946
5,Koffer/Werkzeugkasten,0,10,10,5,0.024886,0.000287,2.742478,1.096991,1.742478,0.096991,verkleinert,0.055663
6,Koffer/Werkzeugkasten,1,10,5,0,0.006863,-0.005058,1.475469,0.590188,0.475469,-0.409812,umgekehrt,0.861911
7,Koffer/Werkzeugkasten,2,10,10,5,0.009407,-0.000543,2.272244,0.908898,1.272244,-0.091102,umgekehrt,0.071608
8,Koffer/Werkzeugkasten,3,10,10,1,0.002885,-0.000114,2.286538,0.914615,1.286538,-0.085385,umgekehrt,0.066368
9,Koffer/Werkzeugkasten,4,10,10,0,0.003075,-0.001639,1.428756,0.571502,0.428756,-0.428498,umgekehrt,0.999397



wrote 5 tables -> exports/size_control_*.csv, length_control_*.csv
